# W&B Leaderboard — Tamper Detection Ablation Study

Queries the Weights & Biases API to generate experiment comparison tables
and export a leaderboard CSV from all completed runs.

In [ ]:
!pip install -q wandb pandas tabulate

import wandb
import pandas as pd
from tabulate import tabulate

# ── Configuration ──
WANDB_PROJECT = 'tamper-detection-ablation'
WANDB_ENTITY = None  # Set to your W&B team/username, or None for default

wandb.login()
api = wandb.Api()
print(f'Connected to W&B. Querying project: {WANDB_PROJECT}')

In [ ]:
# ── Fetch all runs ──
entity = WANDB_ENTITY or api.default_entity
runs = api.runs(f'{entity}/{WANDB_PROJECT}')

print(f'Found {len(runs)} runs\n')

records = []
for run in runs:
    cfg = run.config
    summary = run.summary._json_dict
    
    records.append({
        'experiment': cfg.get('experiment', ''),
        'version': cfg.get('version', ''),
        'change': cfg.get('change', '')[:60],
        'run': cfg.get('run', ''),
        'feature_set': cfg.get('feature_set', ''),
        'encoder': cfg.get('encoder', ''),
        'epochs': cfg.get('epochs', ''),
        'tta': cfg.get('tta', False),
        'pixel_f1': summary.get('pixel_f1', None),
        'pixel_iou': summary.get('pixel_iou', None),
        'pixel_auc': summary.get('pixel_auc', None),
        'pixel_precision': summary.get('pixel_precision', None),
        'pixel_recall': summary.get('pixel_recall', None),
        'image_accuracy': summary.get('image_accuracy', None),
        'image_macro_f1': summary.get('image_macro_f1', None),
        'image_roc_auc': summary.get('image_roc_auc', None),
        'wandb_run_id': run.id,
        'state': run.state,
        'duration_min': run.summary.get('_wandb', {}).get('runtime', 0) / 60,
    })

df = pd.DataFrame(records)
print(f'Collected {len(df)} run records')
df.head()

In [ ]:
# ── Pixel F1 Leaderboard ──
leaderboard = df[df['pixel_f1'].notna()].sort_values('pixel_f1', ascending=False).reset_index(drop=True)
leaderboard.index += 1
leaderboard.index.name = 'Rank'

display_cols = ['version', 'change', 'feature_set', 'pixel_f1', 'pixel_iou', 
                'pixel_auc', 'image_accuracy', 'run', 'state']

print('=' * 80)
print('  PIXEL F1 LEADERBOARD — Tamper Detection Ablation Study')
print('=' * 80)
print()
print(tabulate(leaderboard[display_cols], headers='keys', tablefmt='pipe', 
               floatfmt='.4f', showindex=True))
print()
print(f'Total runs: {len(leaderboard)}')
if len(leaderboard) > 0:
    best = leaderboard.iloc[0]
    print(f'Best: {best["version"]} — Pixel F1 = {best["pixel_f1"]:.4f}, '
          f'IoU = {best["pixel_iou"]:.4f}, Image Acc = {best["image_accuracy"]:.4f}')

In [ ]:
# ── Image Accuracy Leaderboard ──
img_lb = df[df['image_accuracy'].notna()].sort_values('image_accuracy', ascending=False).reset_index(drop=True)
img_lb.index += 1
img_lb.index.name = 'Rank'

img_cols = ['version', 'change', 'image_accuracy', 'image_macro_f1', 
            'image_roc_auc', 'pixel_f1', 'run']

print('=' * 80)
print('  IMAGE ACCURACY LEADERBOARD')
print('=' * 80)
print()
print(tabulate(img_lb[img_cols], headers='keys', tablefmt='pipe', 
               floatfmt='.4f', showindex=True))

In [ ]:
# ── Feature Set Comparison ──
if 'feature_set' in df.columns and df['pixel_f1'].notna().any():
    feat_summary = df[df['pixel_f1'].notna()].groupby('feature_set').agg({
        'pixel_f1': ['mean', 'max', 'count'],
        'pixel_iou': ['mean', 'max'],
        'image_accuracy': ['mean', 'max'],
    }).round(4)
    
    feat_summary.columns = ['F1_mean', 'F1_max', 'runs', 'IoU_mean', 'IoU_max', 
                            'ImgAcc_mean', 'ImgAcc_max']
    feat_summary = feat_summary.sort_values('F1_max', ascending=False)
    
    print('=' * 80)
    print('  FEATURE SET COMPARISON')
    print('=' * 80)
    print()
    print(tabulate(feat_summary, headers='keys', tablefmt='pipe', 
                   floatfmt='.4f', showindex=True))

In [ ]:
# ── Export CSV ──
export_path = 'leaderboard.csv'
leaderboard.to_csv(export_path)
print(f'Leaderboard exported to {export_path}')

# Full results export
full_path = 'all_experiment_results.csv'
df.to_csv(full_path, index=False)
print(f'Full results exported to {full_path}')

# Log to W&B as artifact
try:
    with wandb.init(project=WANDB_PROJECT, name='leaderboard_export', 
                    job_type='analysis', reinit=True):
        artifact = wandb.Artifact('leaderboard', type='results')
        artifact.add_file(export_path)
        artifact.add_file(full_path)
        wandb.log_artifact(artifact)
        
        # Log summary table
        wandb.log({'leaderboard': wandb.Table(dataframe=leaderboard[display_cols])})
    print('Leaderboard logged to W&B')
except Exception as e:
    print(f'W&B artifact upload failed: {e}')

In [ ]:
# ── Run Duration Summary ──
if 'duration_min' in df.columns:
    duration = df[['version', 'run', 'duration_min', 'state']].sort_values('duration_min', ascending=False)
    total_min = duration['duration_min'].sum()
    print(f'\nTotal GPU time: {total_min:.0f} min ({total_min/60:.1f} hours)')
    print(f'Average per run: {total_min/len(duration):.0f} min')
    print()
    print(tabulate(duration, headers='keys', tablefmt='pipe', 
                   floatfmt='.1f', showindex=False))